# Initialization

In [0]:
from pyspark.sql.functions import trim, col
from pyspark.sql.types import StringType
import pyspark.sql.functions as F

#Reading from Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

# Transformations

- Remove rows with missing customer id
- Duplicate customer ids exist so keep latest row with most complete info
- Remove whitespaces from any text columns
- Normalize gender and marital status columns
- Rename column headers

### Removing rows with missing customer ids

In [0]:
df = df.filter(col("cst_id").isNotNull())

### Deduplicating rows so customer id is unique

In [0]:
df = df.groupBy("cst_id", "cst_key").agg(
    F.last("cst_firstname", ignorenulls=True).alias("cst_firstname"),
    F.last("cst_lastname", ignorenulls=True).alias("cst_lastname"),
    F.last("cst_marital_status", ignorenulls=True).alias("cst_marital_status"),
    F.last("cst_gndr", ignorenulls=True).alias("cst_gndr"),
    F.max("cst_create_date").alias("cst_create_date")
)

### Trimming text columns

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Normalizing marital and gender columns

In [0]:
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .when(F.upper(F.col("cst_marital_status")) == "S", "Single")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .when(F.upper(F.col("cst_gndr")) == "F", "Female")
    )
)

### Renaming columns

In [0]:
RENAME_MAPPING = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

for old_name, new_name in RENAME_MAPPING.items():
    df = df.withColumnRenamed(old_name, new_name)

### Sanity check dataframe

In [0]:
df.limit(10).display()

# Writing to Silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")

### Sanity check silver table

In [0]:
%sql

SELECT * FROM workspace.silver.crm_customers LIMIT 10